In [10]:
import numpy as np
import itertools
import re


def extract_lrs(s):
    lr_match = re.search(r'_lr([0-9.eE-]+?)(?:[._]|$)', s)
    lr = lr_match.group(1) if lr_match else None

    ti_lr_match = re.search(r'\.ti([0-9.eE-]+?)(?:[._]|$)', s)
    ti_lr = ti_lr_match.group(1) if ti_lr_match else None

    return lr, ti_lr

def extract_learning_lora_rank(s):
    match = re.search(r'c\.l(\d+)\.', s)
    if match:
        return int(match.group(1))
    else:
        return None


dataset_name2data_root = {
    'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
    'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
    'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
    'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
    'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
    'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
    'avp20': 'data_root/data/real_data/avp/avp-20',
    'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
}
concept2prompt = {
    'crybaby': 'A photo of a crybaby art toy',
    'moodeng': 'A photo of a cute baby hippo',
}
concept2generalprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    
}
concept2initializer = {
    'crybaby': 'toy',
    'moodeng': 'hippo',
    'chiquita': 'girl', 
    'avp': 'glasses',
}

concept2Prprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    'chiquita': 'A photo of a girl',
    'avp': 'A photo of a glasses',
}



In [5]:
base_exps = [
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4",
]

exp_names = []

learning_rates = ["1e-4"]# ["1e-3", "1e-4", "1e-5"]
num_gen_images = [8] # [50] # [8] 
lora_ranks = [1] # [1]
img_types = ["G"] # ["r","g"]
max_train_steps = [0] # [50,200]

target_concept = "chiquita"

base_exp_steps = [3000] # we want to see it fit first
for base_exp in base_exps:
    for base_exp_step in base_exp_steps:
        for lr, num_img, lora_rank, img_type,  max_train_step in itertools.product(
            learning_rates, num_gen_images, lora_ranks, img_types, max_train_steps
        ):
            # print(lr, num_img, lora_rank, img_type, steps)
            
            ul_name  = f'ul{lora_rank}.lr{lr}.n{num_img}.{img_type}'
            exp_name = f"{ul_name}.{target_concept}.obj.s{max_train_step}_{base_exp}.s{base_exp_step}"
            # print(exp_name)
            
            script = f""" python data_preparation.py configs/custom/erase_default.yaml \\
            exp_name="{exp_name}" \\
            MACE.num_gen_images={num_img} \\
            MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}"
 python training.py configs/custom/erase_default.yaml \\
            exp_name="{exp_name}" \\
            MACE.learning_rate={lr} MACE.max_train_steps={max_train_step} \\
            MACE.rank={lora_rank} \\
            MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}"
            """

            
            print(script)
            print(exp_name)
            exp_names += [exp_name]
print(exp_names)

 python data_preparation.py configs/custom/erase_default.yaml \
            exp_name="ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000" \
            MACE.num_gen_images=8 \
            MACE.lora_weight_dir_path="data_root/logs/c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4/checkpoint-3000" \
            MACE.token_embedding_dir_path="data_root/logs/c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4/checkpoint-3000" \
            MACE.input_data_dir="data_root/generated/mace/c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4/checkpoint-3000"
 python training.py configs/custom/erase_default.yaml \
            exp_name="ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000" \
            MACE.learning_rate=1e-4 MACE.max_train_steps=0 \
            MACE.rank=1 \
            MACE.input_data_dir="data_root/generated/mace/c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4/checkpoint-3000" \
            MACE.

In [ ]:

# decoding unlearning - with same hyperparameter

# ul_exp_names = ['ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000']
ul_exp_names = ['ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000']

ul_exp_names = ['ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4.s3000']
ul_exp_names = ['ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4.s3000']
ul_exp_names = ['ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4.s3000', 'ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4.s3000']
ul_exp_names = ['ul1.lr1e-4.n8.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.lr1e-4.n8.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000']

ul_exp_names = ['ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000']









# ul_exp_names = [
#     "ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    
    
#     # "ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4.s3000"
# ]
data_setting = 'full' 
use_ni = True
for ul_exp_name in ul_exp_names:
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
    pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

    if 'moodeng' in exp_name: concept = 'moodeng'
    if 'crybaby' in exp_name: concept = 'crybaby'
    if 'avp' in exp_name: concept = 'avp'
    if 'chiquita' in exp_name: concept = 'chiquita'
    use_ti = 'ti' in exp_name or '-V' in exp_name 
    use_pr = 'pr' in exp_name


    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

            
    if use_ni:
        initializer_token = ''
    elif use_ti:
        initializer_token = concept2initializer[concept]
        
    if use_ti:
        prompt = 'A photo of a v1'
    else:
        prompt = concept2prompt[concept]

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'

    data_root = dataset_name2data_root[dataset_name]
    
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        # if use_ni:
        #     dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name
    
    
    # renaming to check
    re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        re_exp_name += f'_pr0.50'
    re_exp_name += '_lr'
    if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
    if use_ti:
        re_exp_name += f'.ti{str(lr_ti)}'
    re_exp_name += '_f0.5_b1g4'
    
    # print(re_exp_name)
    assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/u{ul_exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        
    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)




        


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=data_root/logs/ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000/LoRA_fusion_model  \
    --instance_data_dir=data_root/data/real_data/chiquita/chiquita-50 \
    --output_dir="data_root/logs/uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000" \
    --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
    --run_note 'uul chiquita50 l4 ti' \
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \
    --class_prompt="A photo of a girl" --class_data_dir="data_root/generated/model/original_pretrained/A photo of a girl/7.50" \
    --learning_rate_lora 5e-4 --learning_rate

In [ ]:
exp_names = [
    "uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000"
]
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_relearn = 'uul' in exp_name
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[1:])
        relearn_exp_name = exp_name
        unlearn_exp_name = relearn_exp_name[1:]
        exp_name = base_exp_name

    
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    cfg_scales = [3.0]
    # steps = [50,100,150,200]
    # for step in steps:
    for step in range(0, 3000+1, 100):
    # for step in [2000]:

        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_unlearn = 'ul' in exp_name and not 'uul' in exp_name
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
                # erase_name = concept
                # if 'VPr' in exp_name: erase_name += 'VPr'
                # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
            if is_unlearn: 
                pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name 
            
            if is_relearn or 'V.ni' in exp_name:
                # relearn is not re-initializing the token (by default)
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]

            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
                
                
            ## hacky .. should change this later
            if is_relearn:
                exp_name = relearn_exp_name
            if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            if 'l0' in exp_name :
                load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti and not is_unlearn:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        


            accelerate launch train_dreambooth_lora.py \
                --pretrained_model_name_or_path='data_root/logs/ul1.lr1e-4.n8.G.chiquita.obj.s10_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000/LoRA_fusion_model'  \
                --instance_data_dir="data_root/data/real_data/dummy" \
                --load_lora_weight_path="data_root/logs/uul1.lr1e-4.n8.G.chiquita.obj.s10_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000/checkpoint-0" \
                --gen_image_path="auto" \
                --output_dir="data_root/logs/gen" \
                --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
                --run_note 'gen img' --wait_weight \
                --num_validation_images 50 \
                --load_token_embedding_path="data_root/logs/uul1.lr1e-4.n8.G.chiquita.obj.s10_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000/che

In [8]:
exp_names = [
    "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
]



    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    cfg_scales = [4.5,6.0,7.5]
    # steps = [50,100,150,200]
    # for step in steps:
    # for step in range(3100, 4000+1, 100):
    for step in [2000]:
    # for step in range(0, 3000+1, 100):

    # for step in range(300, 1001, 100):
        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
            
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                erase_name = concept
                if 'VPr' in exp_name: erase_name += 'VPr'
                pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name
            
            if 'V.ni' in exp_name:
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]



            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
            
            if 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            if 'l0' in exp_name :
                load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        

NameError: name 'concept2initializer' is not defined

In [7]:
import re

def extract_dataname(s):
    match = re.search(r'kv_([^-_]+(?:\d+))[-_]pr', s)
    if match:
        return match.group(1)
    else:
        return None

# Test cases
ul_exp_names = [
    "c.l4.kv_crybaby50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    "c.l4.kv_crybaby3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    "c.l4.kv_crybaby3_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    "c.l4.kv_moodeng50_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000"
]

for exp_name in ul_exp_names:
    dataname = extract_dataname(exp_name)
    print(f"dataname: {dataname}")


dataname: None
dataname: None
dataname: crybaby3
dataname: moodeng50


In [7]:
import re

def extract_lrs(s):
    lr_match = re.search(r'_lr([0-9.eE-]+?)(?:[._]|$)', s)
    lr = lr_match.group(1) if lr_match else None

    ti_lr_match = re.search(r'\.ti([0-9.eE-]+?)(?:[._]|$)', s)
    ti_lr = ti_lr_match.group(1) if ti_lr_match else None

    return lr, ti_lr

def extract_learning_lora_rank(s):
    match = re.search(r'c\.l(\d+)\.', s)
    if match:
        return int(match.group(1))
    else:
        return None

# test cases
ul_exp_names = [
    "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    "c.l1.kv_chiquita50-V_pr0.50_lr3e-4.ti1e-2_f0.5_b1g4.s3000"
]

for exp_name in ul_exp_names:
    lr, ti_lr = extract_lrs(exp_name)
    print(f"lr: {lr}, ti_lr: {ti_lr}")
    
    learning_lora_rank = extract_learning_lora_rank(exp_name)
    print(f"Learning LoRA Rank: {learning_lora_rank}")


lr: 5e-4, ti_lr: 5e-2
Learning LoRA Rank: 4
lr: 3e-4, ti_lr: 1e-2
Learning LoRA Rank: 1


In [69]:

# decoding unlearning - with same hyperparameter
ul_exp_names = [
    "ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    
    
    # "ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4.s3000"
]
data_setting = 'full' 
use_ni = True
for ul_exp_name in ul_exp_names:
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
    pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

    if 'moodeng' in exp_name: concept = 'moodeng'
    if 'crybaby' in exp_name: concept = 'crybaby'
    if 'avp' in exp_name: concept = 'avp'
    if 'chiquita' in exp_name: concept = 'chiquita'
    use_ti = 'ti' in exp_name or '-V' in exp_name 
    use_pr = 'pr' in exp_name


    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

            
    if use_ni:
        initializer_token = ''
    elif use_ti:
        initializer_token = concept2initializer[concept]
        
    if use_ti:
        prompt = 'A photo of a v1'
    else:
        prompt = concept2prompt[concept]

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'

    data_root = dataset_name2data_root[dataset_name]
    
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        # if use_ni:
        #     dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name
    
    
    # renaming to check
    re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        re_exp_name += f'_pr0.50'
    re_exp_name += '_lr'
    if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
    if use_ti:
        re_exp_name += f'.ti{str(lr_ti)}'
    re_exp_name += '_f0.5_b1g4'
    
    # print(re_exp_name)
    assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/u{ul_exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        
    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)




        


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=data_root/logs/ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/LoRA_fusion_model  \
    --instance_data_dir=data_root/data/real_data/chiquita/chiquita-50 \
    --output_dir="data_root/logs/uul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000" \
    --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
    --run_note ' chiquita50 l4 ti' \
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \
    --class_prompt="A photo of a girl" --class_data_dir="data_root/generated/model/original_pretrained/A photo of a girl/7.50" \
    --learning_rate_lora 5e-4 --learning_rat

In [ ]:


concept = 'chiquita' # moodeng
data_setting = 'full' # full
is_relearn = False # True

lora_rank = 4 # 1
use_pr = True
use_ti = True # True 
use_ni = False

# lr_lora = "2.5e-4" # 1e-4
# lr_ti = "1e-2" #  apply only if use_ti
lr_lora_grid = ["5e-4", "1e-4", "5e-5", "1e-5"]
lr_ti_grid   = ["5e-2", "1e-2", "5e-3", "1e-3"]   # only used if use_ti
combos = itertools.product(lr_lora_grid,
                           lr_ti_grid if use_ti else [None])

for lr_lora, lr_ti in combos:

    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

    dataset_name2data_root = {
        'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
        'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
        'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
        'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
        'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
        'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
        'avp20': 'data_root/data/real_data/avp/avp-20',
        'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
    }
    concept2prompt = {
        'crybaby': 'A photo of a crybaby art toy',
        'moodeng': 'A photo of a cute baby hippo',
    }
    concept2generalprompt = {
        'crybaby': 'A photo of a toy',
        'moodeng': 'A photo of a hippo',
        
    }
    concept2initializer = {
        'crybaby': 'toy',
        'moodeng': 'hippo',
        'chiquita': 'girl', 
        'avp': 'glasses',
    }

    concept2Prprompt = {
        'crybaby': 'A photo of a toy',
        'moodeng': 'A photo of a hippo',
        'chiquita': 'A photo of a girl',
        'avp': 'A photo of a glasses',
    }




concept = 'chiquita' # moodeng
data_setting = 'full' # full
is_relearn = False # True

lora_rank = 4 # 1
use_pr = True
use_ti = True # True 
use_ni = False

# lr_lora = "2.5e-4" # 1e-4
# lr_ti = "1e-2" #  apply only if use_ti
lr_lora_grid = ["5e-4", "1e-4", "5e-5", "1e-5"]
lr_ti_grid   = ["5e-2", "1e-2", "5e-3", "1e-3"]   # only used if use_ti
combos = itertools.product(lr_lora_grid,
                           lr_ti_grid if use_ti else [None])

for lr_lora, lr_ti in combos:

    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

    dataset_name2data_root = {
        'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
        'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
        'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
        'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
        'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
        'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
        'avp20': 'data_root/data/real_data/avp/avp-20',
        'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
    }
    concept2prompt = {
        'crybaby': 'A photo of a crybaby art toy',
        'moodeng': 'A photo of a cute baby hippo',
    }
    concept2generalprompt = {
        'crybaby': 'A photo of a toy',
        'moodeng': 'A photo of a hippo',
        
    }
    concept2initializer = {
        'crybaby': 'toy',
        'moodeng': 'hippo',
        'chiquita': 'girl', 
        'avp': 'glasses',
    }

    concept2Prprompt = {
        'crybaby': 'A photo of a toy',
        'moodeng': 'A photo of a hippo',
        'chiquita': 'A photo of a girl',
        'avp': 'A photo of a glasses',
    }


concept = 'chiquita' # moodeng
data_setting = 'full' # full
is_relearn = False # True

lora_rank = 4 # 1
use_pr = True
use_ti = True # True 
use_ni = False

# lr_lora = "2.5e-4" # 1e-4
# lr_ti = "1e-2" #  apply only if use_ti
lr_lora_grid = ["5e-4", "1e-4", "5e-5", "1e-5"]
lr_ti_grid   = ["5e-2", "1e-2", "5e-3", "1e-3"]   # only used if use_ti
combos = itertools.product(lr_lora_grid,
                           lr_ti_grid if use_ti else [None])

for lr_lora, lr_ti in combos:

    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

    dataset_name2data_root = {
        'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
        'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
        'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
        'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
        'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
        'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
        'avp20': 'data_root/data/real_data/avp/avp-20',
        'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
    }
    concept2prompt = {
        'crybaby': 'A photo of a crybaby art toy',
        'moodeng': 'A photo of a cute baby hippo',
    }
    concept2generalprompt = {
        'crybaby': 'A photo of a toy',
        'moodeng': 'A photo of a hippo',
        
    }
    concept2initializer = {
        'crybaby': 'toy',
        'moodeng': 'hippo',
        'chiquita': 'girl', 
        'avp': 'glasses',
    }

    concept2Prprompt = {
        'crybaby': 'A photo of a toy',
        'moodeng': 'A photo of a hippo',
        'chiquita': 'A photo of a girl',
        'avp': 'A photo of a glasses',
    }
    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        prompt = 'A photo of a v1' 
    else:
        prompt = concept2prompt[concept]
        
    pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name

    exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        exp_name += f'_pr0.50'
    exp_name += '_lr'
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += '_f0.5_b1g4'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]



    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        
    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    # print(script)
    print(exp_name)




c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4
c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4


In [ ]:


concept = 'chiquita' # moodeng
data_setting = 'full' # full
is_relearn = False # True

lora_rank = 4 # 1
use_pr = True
use_ti = True # True 
use_ni = False

lr_lora = "2.5e-4" # 1e-4
lr_ti = "1e-2" #  apply only if use_ti



if data_setting == 'fewshot':
    
    if concept == 'avp':
        dataset_name = 'avpS3'
    else:
        dataset_name = f'{concept}U3'
else:
    if concept == 'avp':
        dataset_name = 'avp20'
    else:
        dataset_name = f'{concept}50'

dataset_name2data_root = {
    'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
    'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
    'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
    'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
    'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
    'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
    'avp20': 'data_root/data/real_data/avp/avp-20',
    'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
}
concept2prompt = {
    'crybaby': 'A photo of a crybaby art toy',
    'moodeng': 'A photo of a cute baby hippo',
}
concept2generalprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    
}
concept2initializer = {
    'crybaby': 'toy',
    'moodeng': 'hippo',
    'chiquita': 'girl', 
    'avp': 'glasses',
}

concept2Prprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    'chiquita': 'A photo of a girl',
    'avp': 'A photo of a glasses',
}
# however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
data_root = dataset_name2data_root[dataset_name]
if use_ti:
    prompt = 'A photo of a v1' 
else:
    prompt = concept2prompt[concept]
    
pretrained_path = 'CompVis/stable-diffusion-v1-4' 
if  is_relearn:
    pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

        
if use_ti:
    dataset_name_for_exp = dataset_name + "-V"
    if use_ni:
        dataset_name_for_exp += ".ni"
else: dataset_name_for_exp = dataset_name

exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
if use_pr:
    exp_name += f'_pr0.50'
exp_name += '_lr'
if lora_rank >0: exp_name += f"{str(lr_lora)}"
if use_ti:
    exp_name += f'.ti{str(lr_ti)}'
exp_name += '_f0.5_b1g4'
if is_relearn:
    unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
    exp_name = f'uul.{unlearn_setting}_{exp_name}'
    
if use_ni: initializer_token = ''
else: 
    initializer_token = concept2initializer[concept]



name_tag = ''
if is_relearn: name_tag += 'uul'
name_tag = f'{name_tag} {dataset_name}'
name_tag += f' l{lora_rank}'
if use_ti: 
    # name_tag += f' ti.{lr_ti}'
    name_tag += f' ti'


script = f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path={pretrained_path}  \\
  --instance_data_dir={data_root} \\
  --output_dir="data_root/logs/{exp_name}" \\
  --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
  --train_batch_size=1 --gradient_accumulation_steps=4 \\
  --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
  --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \\
  --run_note '{name_tag}' \\"""
    
    
if use_pr:
    script += f"""
  --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
  --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
    
# Conditional learning rate + TI options
if use_ti:
    
    if lora_rank <= 0:
        script += f"""
  --learning_rate_ti {lr_ti} \\
  --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
  --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
  --placeholder_token="v1" --initializer_token='{initializer_token}'"""
else:
    script += f"""
  --learning_rate {lr_lora}"""

print(script)





accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path=CompVis/stable-diffusion-v1-4  \
  --instance_data_dir=data_root/data/real_data/avp/avp-20 \
  --output_dir="data_root/logs/c.l16.kv_avp20-V_lr2.5e-4.ti1e-2_f0.5_b1g4" \
  --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --lora_rank 16 --target_lora_modules to_k to_v --target_lora_layers cross \
  --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
  --run_note ' avp20 l16 ti' \
  --learning_rate_lora 2.5e-4 --learning_rate_ti 1e-2 \
  --placeholder_token="v1" --initializer_token='glasses'


In [ ]:

accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path=data_root/logs/erase_l1.moodengVPr.object_lr2.5e-4/LoRA_fusion_model  \
  --instance_data_dir=data_root/data/real_data/moodeng/moodeng-50 \
  --output_dir="data_root/logs/uul.l1.moodengVPr.object_c.l0.kv_moodeng50-V.ni_lr.ti1e-2_f0.5_b1g4" \
  --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --lora_rank 0 --target_lora_modules to_k to_v --target_lora_layers cross \
  --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
  --run_note 'uul moodeng50 l0 ti' \
  --learning_rate_ti 1e-2 \
  --placeholder_token="v1" --initializer_token=''

In [ ]:

accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path=data_root/logs/erase_l1.crybabyVPr.object_lr2.5e-4/LoRA_fusion_model  \
  --instance_data_dir=data_root/data/real_data/crybaby/crybaby-50 \
  --output_dir="data_root/logs/uul.l1.crybabyVPr.object_c.l4.kv_crybaby50-V.ni_lr2.5e-4.ti1e-2_f0.5_b1g4" \
  --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
  --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
  --run_note 'uul crybaby50 l4 ti' \
  --learning_rate_lora 2.5e-4 --learning_rate_ti 1e-2 \
  --placeholder_token="v1" --initializer_token=''

In [ ]:
exp_names = [
    "uul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000"
]



    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    cfg_scales = [3.0]
    # steps = [50,100,150,200]
    # for step in steps:
    # for step in range(3100, 4000+1, 100):
    for step in [2000]:
    # for step in range(0, 3000+1, 100):

    # for step in range(300, 1001, 100):
        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
            is_unlearn = 'ul' in exp_name and not 'uul' in exp_name
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                erase_name = concept
                if 'VPr' in exp_name: erase_name += 'VPr'
                pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
            if is_unlearn: 
                pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name 
            
            if 'V.ni' in exp_name:
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]



            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
            
            if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            if 'l0' in exp_name :
                load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti and not is_unlearn:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        


            accelerate launch train_dreambooth_lora.py \
                --pretrained_model_name_or_path='data_root/logs/ul1.lr1e-4.n8.G.s200-chiquita.obj_s2000.c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4/LoRA_fusion_model'  \
                --instance_data_dir="data_root/data/real_data/dummy" \
                --load_lora_weight_path="" \
                --gen_image_path="data_root/generated/model/ul1.lr1e-4.n8.G.s200-chiquita.obj_s2000.c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4" \
                --output_dir="data_root/logs/gen" \
                --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
                --run_note 'gen img' --wait_weight \
                --num_validation_images 50 \
                --cfg_scale 3.00


In [205]:
# decoding exp_name to gneration script 

# exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"

exp_name = "" # 'CompVis/stable-diffusion-v1-4' # 'c.l1.kv_moodeng50-V.ni_lr.ti1e-2_f0.5_b1g4'
# manual_prompt = 'A photo of a toy'# 'A photo of a toy'
# manual_prompt = 'A photo of a hippo'
manual_prompt = ''
use_general_concept = False
# cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

cfg_scales = [3.0]
# steps = [50,100,150,200]
# for step in steps:
# for step in range(3100, 4000+1, 100):
for step in range(0, 3000+1, 250):

# for step in range(300, 1001, 100):
    for cfg in cfg_scales:
      is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
      is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
      
      if 'moodeng' in exp_name: concept = 'moodeng'
      if 'crybaby' in exp_name: concept = 'crybaby'
      if 'avp' in exp_name: concept = 'avp'
      if 'chiquita' in exp_name: concept = 'chiquita'
      
      
      pretrained_path = 'CompVis/stable-diffusion-v1-4'
      if is_relearn:
          erase_name = concept
          if 'VPr' in exp_name: erase_name += 'VPr'
          pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"

      use_ti = 'ti' in exp_name or '-V' in exp_name
      
      if 'V.ni' in exp_name:
         initializer_token = ''
      elif use_ti:
        initializer_token = concept2initializer[concept]



      if manual_prompt:
          prompt = manual_prompt
      elif use_general_concept:
         prompt = concept2generalprompt[concept]
      
      elif use_ti:
        prompt = 'A photo of a v1'
      else:
        prompt = concept2prompt[concept]
      
      if 'erase' in exp_name or exp_name == 'original_pretrained': 
        load_lora_weight_path = ''
        gen_image_path = f"data_root/generated/model/{exp_name}"
      else:
        load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
        gen_image_path = 'auto'
        
      if 'l0' in exp_name :
          load_lora_weight_path = ''
      
      script = f"""
      accelerate launch train_dreambooth_lora.py \\
        --pretrained_model_name_or_path='{pretrained_path}'  \\
        --instance_data_dir="data_root/data/real_data/dummy" \\
        --load_lora_weight_path="{load_lora_weight_path}" \\
        --gen_image_path="{gen_image_path}" \\
        --output_dir="data_root/logs/gen" \\
        --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
        --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
        --run_note 'gen img' \\
        --num_validation_images 50 \\"""
        
              
      if use_ti:
          script += f"""
        --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
        --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

      script += f"""
        --cfg_scale {cfg:.2f}"""
 
      print(script) 
      


      accelerate launch train_dreambooth_lora.py \
        --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
        --instance_data_dir="data_root/data/real_data/dummy" \
        --load_lora_weight_path="data_root/logs//checkpoint-0" \
        --gen_image_path="auto" \
        --output_dir="data_root/logs/gen" \
        --validation_prompt="A photo of a crybaby art toy" --instance_prompt="A photo of a crybaby art toy" \
        --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
        --run_note 'gen img' \
        --num_validation_images 50 \
        --cfg_scale 3.00

      accelerate launch train_dreambooth_lora.py \
        --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
        --instance_data_dir="data_root/data/real_data/dummy" \
        --load_lora_weight_path="data_root/logs//checkpoint-250" \
        --gen_image_path="auto" \
        --output_dir="data_root/logs/gen" \
        --validation_prompt="A photo of a cryb

In [14]:
# manual_prompts = [f"A photo of a {c}" for c in ['cat', 'fish', 'bird', 'house', 'mountain']]
lora_rank =1
manual_prompts = ["A photo of a girl","A photo of a person"]
for manual_prompt in manual_prompts:
    exp_name = "original_pretrained"
    use_general_concept = False
    cfg_scales = [3.0, 4.5, 6.0, 7.5]

    for step in range(0, 1, 250):
        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_relearn = ('uul' in exp_name) or ('erase' in exp_name)

            concept = 'moodeng' if 'moodeng' in exp_name else 'crybaby'
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                if concept == 'crybaby':
                    pretrained_path = "data_root/logs/erase_l1.crybaby.object_lr2.5e-4/LoRA_fusion_model"
                elif concept == 'moodeng':
                    pretrained_path = "data_root/logs/erase_l1.moodeng.object_lr2.5e-4/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name

            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            elif use_ti:
                prompt = 'A photo of a v1'
                initializer_token = 'hippo' if concept == 'moodeng' else 'toy'
            else:
                prompt = concept2prompt[concept]

            if 'erase' in exp_name or exp_name == 'original_pretrained':
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path = f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'

            script = f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path='{pretrained_path}'  \\
  --instance_data_dir="data_root/data/real_data/dummy" \\
  --load_lora_weight_path="{load_lora_weight_path}" \\
  --gen_image_path="{gen_image_path}" \\
  --output_dir="data_root/logs/gen" \\
  --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
  --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
  --run_note 'gen img' \\
  --num_validation_images 50 \\"""

            if use_ti:
                script += f"""
  --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
  --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
  --cfg_scale {cfg:.2f}"""

            print(script)



accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
  --instance_data_dir="data_root/data/real_data/dummy" \
  --load_lora_weight_path="" \
  --gen_image_path="data_root/generated/model/original_pretrained" \
  --output_dir="data_root/logs/gen" \
  --validation_prompt="A photo of a girl" --instance_prompt="A photo of a girl" \
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
  --run_note 'gen img' \
  --num_validation_images 50 \
  --cfg_scale 3.00

accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
  --instance_data_dir="data_root/data/real_data/dummy" \
  --load_lora_weight_path="" \
  --gen_image_path="data_root/generated/model/original_pretrained" \
  --output_dir="data_root/logs/gen" \
  --validation_prompt="A photo of a girl" --instance_prompt="A photo of a girl" \
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_la

In [ ]:
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \\
  --output_dir="data_root/logs/gen" \\
  --validation_prompt="A photo of a cute baby hippo" \\
  --instance_prompt="A photo of a cute baby hippo" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale {cfg:.2f} \\
  --run_note "erased a cute baby hippo"

In [ ]:
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()
for step in range(0, 4001, 250):
    for cfg in cfg_scales:

      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr1e-4_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a crybaby art toy" \\
    --instance_prompt="A photo of a crybaby art toy" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true crybaby"\n""")
      
      
      
    #     print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \
    # --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr1e-4_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of a crybaby art toy" \\
    # --instance_prompt="A photo of a crybaby art toy" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true crybaby"\n""")
      

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr1e-4_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a crybaby art toy" \
    --instance_prompt="A photo of a crybaby art toy" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.00 \
    --run_note "true crybaby"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr1e-4_b1g4/checkpoint-250" \
    --validati

In [5]:
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

for step in range(0, 4001, 250):
    for cfg in cfg_scales:
        print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)
  
        print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)
          
        print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a cute baby hippo" \\
    --instance_prompt="A photo of a cute baby hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
   """)



  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
  

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="a

In [6]:
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

for step in range(0, 4001, 250):
    for cfg in cfg_scales:
        print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)
  
        print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)
          
        print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a cute baby hippo" \\
    --instance_prompt="A photo of a cute baby hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
   """)



  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
  

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_m

In [ ]:
cfg_scales = np.arange(1.0,9.5, 0.5).tolist()

for step in range(3000, 4001, 500):
    for cfg in cfg_scales:
        print(f"""

In [12]:
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()
for step in range(0, 4001, 250):
    for cfg in cfg_scales:

    #   print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of a cute baby hippo" \\
    # --instance_prompt="A photo of a cute baby hippo" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true moodeng"\n""")
      
      
  #     print(f"""
  # accelerate launch train_dreambooth_lora.py \\
  #   --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
  #   --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  #   --gen_image_path="auto" \\
  #   --output_dir="data_root/logs/gen" \\
  #   --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --validation_prompt="A photo of a v1" \\
  #   --instance_prompt="A photo of a v1" \\
  #   --placeholder_token="v1" --initializer_token="hippo" \\
  #   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  #   --num_validation_images 50 \\
  #   --cfg_scale {cfg} \\
  #   --run_note "gen image"
  # """)
      
  #     print(f"""
  # accelerate launch train_dreambooth_lora.py \\
  #   --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
  #   --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  #   --gen_image_path="auto" \\
  #   --output_dir="data_root/logs/gen" \\
  #   --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
  #   --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
  #   --validation_prompt="A photo of a v1" \\
  #   --instance_prompt="A photo of a v1" \\
  #   --placeholder_token="v1" --initializer_token="hippo" \\
  #   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  #   --num_validation_images 50 \\
  #   --cfg_scale {cfg} \\
  #   --run_note "gen image"
  # """)
      
      
  #     print(f"""
  # accelerate launch train_dreambooth_lora.py \\
  #   --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
  #   --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  #   --gen_image_path="auto" \\
  #   --output_dir="data_root/logs/gen" \\
  #   --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodeng50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
  #   --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodeng50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
  #   --validation_prompt="A photo of a cute baby hippo" \\
  #   --instance_prompt="A photo of a cute baby hippo" \\
  #   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  #   --num_validation_images 50 \\
  #   --cfg_scale {cfg} \\
  #   --run_note "gen image"
  #  """)


      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a cute baby hippo" \\
    --instance_prompt="A photo of a cute baby hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
   """)




  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3_lr1e-4_f0.5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a cute baby hippo" \
    --instance_prompt="A photo of a cute baby hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
   

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \
    --instance_data_dir="data_root/data/real_

In [3]:

# erased
cfg_values = np.arange(1.0, 9.5, 0.5)  # includes 9.0


template = """accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \\
  --output_dir="data_root/logs/gen" \\
  --validation_prompt="A photo of a cute baby hippo" \\
  --instance_prompt="A photo of a cute baby hippo" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale {cfg:.2f} \\
  --run_note "erased a cute baby hippo"
"""


# template = """accelerate launch train_dreambooth_lora.py \\
#   --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
#   --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
#   --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \\
#   --output_dir="data_root/logs/gen" \\
#   --validation_prompt="A photo of moodeng" \\
#   --instance_prompt="A photo of moodeng" \\
#   --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
#   --num_validation_images 50 \\
#   --cfg_scale {cfg:.2f} \\
#   --run_note "erased moodeng"
# """

# template = """accelerate launch train_dreambooth_lora.py \\
#   --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
#   --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
#   --gen_image_path="data_root/generated/model/erase_crybaby.object_lr2.5e-4" \\
#   --output_dir="data_root/logs/gen" \\
#   --validation_prompt="A photo of a crybaby art toy" \\
#   --instance_prompt="A photo of a crybaby art toy" \\
#   --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
#   --num_validation_images 1000 \\
#   --cfg_scale {cfg:.2f} \\
#   --run_note "erased crybaby"
# """


for cfg in cfg_values:
    print(template.format(cfg=cfg))
    print()  # 

accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
  --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \
  --output_dir="data_root/logs/gen" \
  --validation_prompt="A photo of a cute baby hippo" \
  --instance_prompt="A photo of a cute baby hippo" \
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
  --num_validation_images 50 \
  --cfg_scale 1.00 \
  --run_note "erased a cute baby hippo"


accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
  --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \
  --output_dir="data_root/logs/gen" \
  --validation_prompt="A photo of a cute baby hippo" \


In [25]:
cfg_scales = np.arange(1.0, 9.5, 0.5)  # Includes 9.0
for step in range(0,4001,250):
  for cfg in cfg_scales:
    #   print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of a cute baby hippo" \\
    # --instance_prompt="A photo of a cute baby hippo" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true moodeng"\n""")
      
    
    
    
  #     print(f"""
  # accelerate launch train_dreambooth_lora.py \\
  #   --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  #   --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  #   --gen_image_path="auto" \\
  #   --output_dir="data_root/logs/gen" \\
  #   --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --load_token_embedding_path="data_root/logs/c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --validation_prompt="A photo of a v1" \\
  #   --instance_prompt="A photo of a v1" \\
  #   --placeholder_token="v1" --initializer_token="hippo" \\
  #   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  #   --num_validation_images 50 \\
  #   --cfg_scale {cfg} \\
  #   --run_note "gen image"
  # """)  

      
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)



  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/c.l4.kv_moodeng50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.0 \
    --run_note "gen image"
  

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_pat

In [22]:
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()
for step in range(0, 4001, 250):
    for cfg in cfg_scales:

      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a crybaby art toy" \\
    --instance_prompt="A photo of a crybaby art toy" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true crybaby"\n""")
      
      
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)
      
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)
      
      
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a crybaby art toy" \\
    --instance_prompt="A photo of a crybaby art toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
   """)


      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)



      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l2.5e-4_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)


accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby50_lr2.5e-4_f0.5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a crybaby art toy" \
    --instance_prompt="A photo of a crybaby art toy" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.00 \
    --run_note "true crybaby"


  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_cr

In [ ]:
# w/o special token
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()
for step in range(0, 4001, 250):
    for cfg in cfg_scales:
    #     print(f"""
    # accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of moodeng" \\
    # --instance_prompt="A photo of moodeng" \\
    # --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg} \\
    # --run_note "reocovered w/o special token"
    # """)
        
        
        print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of moodeng" \\
  --instance_prompt="A photo of moodeng" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale {cfg} \\
  --run_note "reocovered w/o special token"
""")
            

        
        


accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \
  --gen_image_path="auto" \
  --output_dir="data_root/logs/gen" \
  --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
  --validation_prompt="A photo of moodeng" \
  --instance_prompt="A photo of moodeng" \
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
  --num_validation_images 50 \
    --cfg_scale 3.0 \
  --run_note "reocovered w/o special token"


accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \
  --gen_image_path="auto" \
  --output_dir="data_root/logs/gen" \
  --load_lora_weight_path="data_root/l

In [19]:
cfg_scales = np.arange(3.0,3.5, 0.5).tolist()
for step in range(1000, 4001, 250):
    for cfg in cfg_scales:
  #           print(f"""
  # accelerate launch train_dreambooth_lora.py \\
  #   --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  #   --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
  #   --gen_image_path="auto" \\
  #   --output_dir="data_root/logs/gen" \\
  #   --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --load_token_embedding_path="data_root/logs/c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --validation_prompt="A photo of a v1" \\
  #   --instance_prompt="A photo of a v1" \\
  #   --placeholder_token="v1" --initializer_token="toy" \\
  #   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  #   --num_validation_images 50 \\
  #   --cfg_scale {cfg} \\
  #   --run_note "gen image"
  # """)



  
  #     print(f"""
  # accelerate launch train_dreambooth_lora.py \\
  #   --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
  #   --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
  #   --gen_image_path="auto" \\
  #   --output_dir="data_root/logs/gen" \\
  #   --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  #   --validation_prompt="A photo of a v1" \\
  #   --instance_prompt="A photo of a v1" \\
  #   --placeholder_token="v1" --initializer_token="toy" \\
  #   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  #   --num_validation_images 50 \\
  #   --cfg_scale {cfg} \\
  #   --run_note "gen image"
  # """)




      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a crybaby art toy" \\
    --instance_prompt="A photo of a crybaby art toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
   """)



  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-1000" \
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybaby50_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-1000" \
    --validation_prompt="A photo of a crybaby art toy" \
    --instance_prompt="A photo of a crybaby art toy" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
   

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \
    --instance_data_d

In [7]:
import numpy as np

cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

for step in range(1500, 2001, 500):
    for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-seen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-seen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-1500" \
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-1500" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="toy" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/

In [4]:

cfg_scales = np.arange(3.0,3.5, 0.5).tolist()
for step in range(0, 1001, 500):
    for cfg in cfg_scales:
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)



  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/c.l1.kv_crybabyS3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="toy" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
  

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
    --gen_ima

In [20]:
cfg_scales = np.arange(1.0, 6.0 + 0.5, 0.5).tolist()
for step in range(0, 3001, 500):
    for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng-sd-cbh.object_lr2.5e-4/LoRA_fusion_model" \\
    --instance_data_dir="data_root/data/real_data/moodeng/sd/moodeng-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng-sd-cbh.object_c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng-sd-cbh.object_c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng-sd-cbh.object_lr2.5e-4/LoRA_fusion_model" \
    --instance_data_dir="data_root/data/real_data/moodeng/sd/moodeng-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_moodeng-sd-cbh.object_c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/uul_moodeng-sd-cbh.object_c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.0 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/

In [21]:
cfg_scales = np.arange(1.0, 6.0 + 0.5, 0.5).tolist()
for step in range(0, 3001, 500):
    for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \\
    --instance_data_dir="data_root/data/real_data/moodeng/sd/moodeng-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
    --instance_data_dir="data_root/data/real_data/moodeng/sd/moodeng-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodeng.sd.U3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.0 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
    --instance_data_dir="data_root/data/real_data/moodeng/sd/moodeng-unsee

In [9]:
cfg_scales = np.arange(3.0, 3.0 + 0.5, 0.5).tolist()
for step in range(3000, 4001, 250):
    for cfg in cfg_scales:
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)


  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-3000" \
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-3000" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
  

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/Lo

In [ ]:

# erased
cfg_values = np.arange(1.0, 9.5, 0.5)  # includes 9.0


template = """accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \\
  --output_dir="data_root/logs/gen" \\
  --validation_prompt="A photo of a cute baby hippo" \\
  --instance_prompt="A photo of a cute baby hippo" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale {cfg:.2f} \\
  --run_note "erased a cute baby hippo"
"""


# template = """accelerate launch train_dreambooth_lora.py \\
#   --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model"  \\
#   --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
#   --gen_image_path="data_root/generated/model/erase_moodeng.object_lr2.5e-4" \\
#   --output_dir="data_root/logs/gen" \\
#   --validation_prompt="A photo of moodeng" \\
#   --instance_prompt="A photo of moodeng" \\
#   --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
#   --num_validation_images 50 \\
#   --cfg_scale {cfg:.2f} \\
#   --run_note "erased moodeng"
# """

# template = """accelerate launch train_dreambooth_lora.py \\
#   --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model"  \\
#   --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
#   --gen_image_path="data_root/generated/model/erase_crybaby.object_lr2.5e-4" \\
#   --output_dir="data_root/logs/gen" \\
#   --validation_prompt="A photo of a crybaby art toy" \\
#   --instance_prompt="A photo of a crybaby art toy" \\
#   --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
#   --num_validation_images 1000 \\
#   --cfg_scale {cfg:.2f} \\
#   --run_note "erased crybaby"
# """


for cfg in cfg_values:
    print(template.format(cfg=cfg))
    print()  # 

NameError: name 'np' is not defined

In [ ]:
# general
cfg_values = np.arange(1.0, 9.5, 0.5)  # From 1.0 to 9.0 inclusive

# template = """accelerate launch train_dreambooth_lora.py \\
#   --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
#   --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
#   --gen_image_path="data_root/generated/general_concepts" \\
#   --output_dir="data_root/logs/gen" \\
#   --validation_prompt="A photo of a toy" \\
#   --instance_prompt="A photo of a toy" \\
#   --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
#   --num_validation_images 50 \\
#   --cfg_scale {cfg:.2f} \\
#   --run_note "original toy"
# """

template = """accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
  --gen_image_path="data_root/generated/general_concepts" \\
  --output_dir="data_root/logs/gen" \\
  --validation_prompt="A photo of a hippo" \\
  --instance_prompt="A photo of a hippo" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale {cfg:.2f} \\
  --run_note "original hippo"
"""

for cfg in cfg_values:
    print(template.format(cfg=cfg))
    print()


In [7]:
cfg_scales = np.arange(1.0, 6.0, 0.5)  # Includes 9.0
for step in range(2500,3500,500):
  for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-2500" \
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-2500" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.00 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.

In [ ]:
cfg_scales = np.arange(1.0, 6.0, 0.5)  # Includes 9.0
for step in range(1500,3500,500):
  for cfg in cfg_scales:
      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a cute baby hippo" \\
    --instance_prompt="A photo of a cute baby hippo" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true moodeng"\n""")
      

    #   print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50.sks_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of sks hippo" \\
    # --instance_prompt="A photo of sks hippo" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true moodeng"\n""")
      

  

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoint-1500" \
    --validation_prompt="A photo of a cute baby hippo" \
    --instance_prompt="A photo of a cute baby hippo" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.00 \
    --run_note "true moodeng"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoin

In [4]:

cfg_scales = np.arange(1.0, 9.5, 0.5)  # Includes 9.0
for step in range(3500,4500,500):
  for cfg in cfg_scales:
      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a crybaby art toy" \\
    --instance_prompt="A photo of a crybaby art toy" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true crybaby"\n""")


accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3500" \
    --validation_prompt="A photo of a crybaby art toy" \
    --instance_prompt="A photo of a crybaby art toy" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.00 \
    --run_note "true crybaby"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3500" 

In [6]:
cfg_scales = np.arange(1.0, 9.5, 0.5)  # Includes 9.0
for step in range(1500,2000,500):
  for cfg in cfg_scales:
      
    #   print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of moodeng" \\
    # --instance_prompt="A photo of moodeng" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true moodeng"\n""")
    
    
      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a cute baby hippo" \\
    --instance_prompt="A photo of a cute baby hippo" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true moodeng"\n""")
      

    #   print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50.sks_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of sks hippo" \\
    # --instance_prompt="A photo of sks hippo" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true moodeng"\n""")
      

  

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoint-1500" \
    --validation_prompt="A photo of a cute baby hippo" \
    --instance_prompt="A photo of a cute baby hippo" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.00 \
    --run_note "true moodeng"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-sd-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoin

In [5]:
cfg_scales = np.arange(1.0, 6.0, 0.5)  # Includes 9.0
for step in range(2000,3500,500):
  for cfg in cfg_scales:
    #   print(f"""accelerate launch train_dreambooth_lora.py \\
    # --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    # --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    # --gen_image_path="auto" \\
    # --output_dir="data_root/logs/gen" \\
    # --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50.cbh_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    # --validation_prompt="A photo of a cute baby hippo" \\
    # --instance_prompt="A photo of a cute baby hippo" \\
    # --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    # --num_validation_images 50 \\
    # --cfg_scale {cfg:.2f} \\
    # --run_note "true moodeng"\n""")
      

      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50.sks_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of sks hippo" \\
    --instance_prompt="A photo of sks hippo" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true moodeng"\n""")
      

  

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50.sks_lr2.5e-4_f0.5_b1g4/checkpoint-2000" \
    --validation_prompt="A photo of sks hippo" \
    --instance_prompt="A photo of sks hippo" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.00 \
    --run_note "true moodeng"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50.sks_lr2.5e-4_f0.5_b1g4/checkpoint-2000" \
    --valida

In [1]:
import numpy as np


cfg_scales = np.arange(1.0, 9.5, 0.5)  # Includes 9.0
for step in range(0,3500,500):
  for cfg in cfg_scales:
      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of moodeng" \\
    --instance_prompt="A photo of moodeng" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 300 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true moodeng"\n""")


accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50_lr2.5e-4_f0.5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of moodeng" \
    --instance_prompt="A photo of moodeng" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 300 \
    --cfg_scale 1.00 \
    --run_note "true moodeng"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_moodeng-50_lr2.5e-4_f0.5_b1g4/checkpoint-0" \
    --validation_prompt="A ph

In [16]:
import numpy as np

cfg_scales = np.arange(3.0, 3.5, 0.5)  # Includes 9.0
for step in range(3000,4001,250):
  for cfg in cfg_scales:
      print(f"""accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby-50_lr2.5e-4_f0.5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a crybaby art toy" \\
    --instance_prompt="A photo of a crybaby art toy" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg:.2f} \\
    --run_note "true crybaby"\n""")


accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000" \
    --validation_prompt="A photo of a crybaby art toy" \
    --instance_prompt="A photo of a crybaby art toy" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.00 \
    --run_note "true crybaby"

accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-50" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l4.kv_crybaby-50_lr2.5e-4_f0.5_b1g4/checkpoint-3250" \
    

In [10]:
cfg_scales = np.arange(1.0, 9.0 + 0.5, 0.5).tolist()
for step in range(0, 2001, 250):
    for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-0" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 1.0 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/Lo

In [8]:
cfg_scales = np.arange(3.0, 3.0 + 0.5, 0.5).tolist()
for step in range(2000, 4001, 250):
    for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="hippo" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-2000" \
    --load_token_embedding_path="data_root/logs/c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-2000" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="hippo" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \

In [9]:
import numpy as np

cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

for step in range(1750, 4001, 500):
    for cfg in cfg_scales:
        print(f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
    """)



    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-1750" \
    --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-1750" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="toy" \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
    

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-

In [13]:

cfg_scales = np.arange(3.0, 3.5, 0.5).tolist()

for step in range(3000, 4001, 250):
    for cfg in cfg_scales:
      print(f"""
  accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
    --gen_image_path="auto" \\
    --output_dir="data_root/logs/gen" \\
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --load_token_embedding_path="data_root/logs/c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
    --validation_prompt="A photo of a v1" \\
    --instance_prompt="A photo of a v1" \\
    --placeholder_token="v1" --initializer_token="toy" \\
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
    --num_validation_images 50 \\
    --cfg_scale {cfg} \\
    --run_note "gen image"
  """)



  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
    --gen_image_path="auto" \
    --output_dir="data_root/logs/gen" \
    --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-3000" \
    --load_token_embedding_path="data_root/logs/c.l1.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-3000" \
    --validation_prompt="A photo of a v1" \
    --instance_prompt="A photo of a v1" \
    --placeholder_token="v1" --initializer_token="toy" \
    --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
    --num_validation_images 50 \
    --cfg_scale 3.0 \
    --run_note "gen image"
  

  accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
    --g

In [5]:
# w/o special token
for step in range(3500, 4001, 500):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a crybaby art toy" \\
  --instance_prompt="A photo of a crybaby art toy" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale 3.00 \\
  --run_note "reocovered w/o special token"
""")



accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
  --gen_image_path="auto" \
  --output_dir="data_root/logs/gen" \
  --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l1.kv_crybabyU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-3500" \
  --validation_prompt="A photo of a crybaby art toy" \
  --instance_prompt="A photo of a crybaby art toy" \
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
  --num_validation_images 50 \
  --cfg_scale 3.00 \
  --run_note "reocovered w/o special token"


accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
  --gen_image_path="auto" \
  --output_dir="data_root/logs/gen" \
  --load_lora_we

In [6]:
# w/o special token
for step in range(3500, 4001, 500):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a crybaby art toy" \\
  --instance_prompt="A photo of a crybaby art toy" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 50 \\
  --cfg_scale 3.00 \\
  --run_note "few-shot fine tuned w/o special token"
""")



accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
  --gen_image_path="auto" \
  --output_dir="data_root/logs/gen" \
  --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-3500" \
  --validation_prompt="A photo of a crybaby art toy" \
  --instance_prompt="A photo of a crybaby art toy" \
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
  --num_validation_images 50 \
  --cfg_scale 3.00 \
  --run_note "few-shot fine tuned w/o special token"


accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \
  --gen_image_path="auto" \
  --output_dir="data_root/logs/gen" \
  --load_lora_weight_path="data_root/logs/c.l1.kv_crybabyU3_f0.5_lr.ti1e-2.l5e-5_b1g4/chec

In [ ]:
# w/o special token
for step in range(0, 2001, 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of moodeng" \\
  --instance_prompt="A photo of moodeng" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "reocovered w/o special token"
""")


In [ ]:
# w/o special token
for step in range(0, 2001, 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/c.l1.kv_moodengU3_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of moodeng" \\
  --instance_prompt="A photo of moodeng" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "reocovered w/o special token"
""")


In [ ]:
for step in range(0, 2001, 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_crybaby.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --load_token_embedding_path="data_root/logs/uul_crybaby.object_c.l4.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a v1" \\
  --instance_prompt="A photo of a v1" \\
  --placeholder_token="v1" --initializer_token="toy" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "gen image"
""")


In [ ]:
# unseen-sd
for step in range(0,2001 , 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3sd-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3sd-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a v1" \\
  --instance_prompt="A photo of a v1" \\
  --placeholder_token="v1" --initializer_token="hippo" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "gen image"
""")


In [ ]:
for step in range(0,2001 , 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a v1" \\
  --instance_prompt="A photo of a v1" \\
  --placeholder_token="v1" --initializer_token="hippo" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "gen image"
""")


In [ ]:
for step in range(1500, 2001, 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  --instance_data_dir="data_root/data/real_data/crybaby/crybaby-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/c.l4.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --load_token_embedding_path="data_root/logs/c.l4.kv_crybabyU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a v1" \\
  --instance_prompt="A photo of a v1" \\
  --placeholder_token="v1" --initializer_token="toy" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "gen image"
""")


In [ ]:
for step in range(0, 4501, 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="data_root/logs/erase_moodeng.object_lr2.5e-4/LoRA_fusion_model" \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --load_token_embedding_path="data_root/logs/uul_moodeng.object_c.l1.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a v1" \\
  --instance_prompt="A photo of a v1" \\
  --placeholder_token="v1" --initializer_token="hippo" \\
  --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "gen image"
""")


In [ ]:
for step in range(1500, 2001, 250):
    print(f"""
accelerate launch train_dreambooth_lora.py \\
  --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \\
  --instance_data_dir="data_root/data/real_data/moodeng/moodeng-unseen-3" \\
  --gen_image_path="auto" \\
  --output_dir="data_root/logs/gen" \\
  --load_lora_weight_path="data_root/logs/c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --load_token_embedding_path="data_root/logs/c.l4.kv_moodengU3-V_f0.5_lr.ti1e-2.l5e-5_b1g4/checkpoint-{step}" \\
  --validation_prompt="A photo of a v1" \\
  --instance_prompt="A photo of a v1" \\
  --placeholder_token="v1" --initializer_token="hippo" \\
  --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \\
  --num_validation_images 1000 \\
  --run_note "gen image"
""")
